In [1]:
import pandas as pd
import numpy as np
import optuna
import plotly.express as px

import xgboost as xgb
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


In [2]:
df = pd.read_csv("/kaggle/input/property-val-dataset/train.csv")
df['date'] = pd.to_datetime(df['date'])
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month
df['day_of_month'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek

df['effective_built_year'] = df.apply(lambda x: x['yr_renovated'] if x['yr_renovated'] > 0 else x['yr_built'], axis=1)
df['house_age'] = df['sale_year'] - df['effective_built_year']

zip_map = df.groupby('zipcode').apply(lambda x: x['price'].median()).to_dict()
df['zip_index'] = df['zipcode'].map(zip_map)

/tmp/ipykernel_17/989552771.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  zip_map = df.groupby('zipcode').apply(lambda x: x['price'].median()).to_dict()


In [3]:
X = df.drop(columns=['id', 'date','zipcode',"price"])
y = df['price']

# Convert once
dtrain = xgb.DMatrix(X.values, label=y.values)

In [4]:
def objective(trial):
    params = {
        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'eval_metric': 'rmse',
        'verbosity': 0,
        'nthread': -1,  # while using GPU remove it

        'n_estimators': 5000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),

        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        'max_depth': trial.suggest_int('max_depth', 3, 12),

        'subsample': trial.suggest_float('subsample', 0.7, 1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 20.0, log=True),

        'min_child_weight': trial.suggest_int('min_child_weight', 1, 15),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),

        #'max_bin': trial.suggest_categorical('max_bin', [256, 512]),
        'max_bin' : 512
    }

    if params['grow_policy'] == 'lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 32, 512)

    cv_results = xgb.cv(
        params,
        dtrain,
        num_boost_round=params['n_estimators'],
        nfold=5,
        early_stopping_rounds=75,
        seed=42,
        verbose_eval=False
    )

    return cv_results['test-rmse-mean'].min()

# Run the detailed study
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=200) # Need more trials for this larger space

print("Best RMSE:", study.best_value)
print("Best Params:", study.best_params)

[I 2026-01-05 14:48:50,865] A new study created in memory with name: no-name-776ff264-5cea-4972-9e78-47a278aeb96e
[I 2026-01-05 14:49:07,990] Trial 0 finished with value: 114887.77256173178 and parameters: {'learning_rate': 0.06526735295184805, 'grow_policy': 'lossguide', 'max_depth': 8, 'subsample': 0.7654009545396185, 'colsample_bytree': 0.9328901610767458, 'reg_alpha': 0.005778780807520928, 'reg_lambda': 3.8563548590892944e-07, 'min_child_weight': 3, 'gamma': 0.10627257329413899, 'max_leaves': 186}. Best is trial 0 with value: 114887.77256173178.
[I 2026-01-05 14:49:51,975] Trial 1 finished with value: 118253.87568993548 and parameters: {'learning_rate': 0.04252938972482163, 'grow_policy': 'depthwise', 'max_depth': 11, 'subsample': 0.9586103252687714, 'colsample_bytree': 0.9229047877371873, 'reg_alpha': 0.01933190688407285, 'reg_lambda': 0.05791220922696941, 'min_child_weight': 13, 'gamma': 3.6988533795210586e-08}. Best is trial 0 with value: 114887.77256173178.
[I 2026-01-05 14:53:

Best RMSE: 107575.09451405979
Best Params: {'learning_rate': 0.02647226088861122, 'grow_policy': 'lossguide', 'max_depth': 4, 'subsample': 0.7004640052324429, 'colsample_bytree': 0.7471553472650564, 'reg_alpha': 0.05964272035816702, 'reg_lambda': 2.904248236928663e-08, 'min_child_weight': 1, 'gamma': 1.5384227833295443e-08, 'max_leaves': 197}


In [5]:
# 1. Extract the best hyperparameters from the study
best_params = study.best_params

# 2. Add the necessary fixed parameters
# We ensure GPU is enabled and the objective is set correctly
best_params['objective'] = 'reg:squarederror'
best_params['tree_method'] = 'auto' # gpu_hist
best_params['eval_metric'] = 'rmse'

if 'n_estimators' not in best_params:
    best_params['n_estimators'] = 2500

final_model = xgb.XGBRegressor(**best_params)

def eval(y_test,y_pred) :
  print("MSE : ",mean_squared_error(y_test,y_pred))
  print("RMSE : ",mean_squared_error(y_test,y_pred)**0.5)
  print("R2 : ",r2_score(y_test,y_pred))
  return

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  12756445184.0
RMSE :  112944.43405498122
R2 :  0.8983457684516907


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  16515479552.0
RMSE :  128512.56573580655
R2 :  0.886894941329956


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=99)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  13048818688.0
RMSE :  114231.42600878271
R2 :  0.9115065932273865


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=7)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  10253496320.0
RMSE :  101259.54927808043
R2 :  0.915462076663971


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=538)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
eval(y_test,y_pred)

MSE :  12140137472.0
RMSE :  110182.29200738203
R2 :  0.8958529829978943


In [10]:
print(best_params)

{'learning_rate': 0.02647226088861122, 'grow_policy': 'lossguide', 'max_depth': 4, 'subsample': 0.7004640052324429, 'colsample_bytree': 0.7471553472650564, 'reg_alpha': 0.05964272035816702, 'reg_lambda': 2.904248236928663e-08, 'min_child_weight': 1, 'gamma': 1.5384227833295443e-08, 'max_leaves': 197, 'objective': 'reg:squarederror', 'tree_method': 'auto', 'eval_metric': 'rmse', 'n_estimators': 2500}
